# 5. End-to-end RAG evaluation

Run the checked-in evaluation questions through the full pipeline and compare predicted categories with labeled categories. The `unknown` example is excluded from category accuracy because it is an unsupported-answer case.

In [ ]:
from pathlib import Path
import json
import os

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "ingestion").is_dir())
os.chdir(ROOT)

from app.rag.pipeline import RAGPipeline

with (ROOT / "data/evaluation/questions.json").open(encoding="utf-8") as handle:
    questions = json.load(handle)

pipeline = RAGPipeline()
results = []
for item in questions:
    result = pipeline.ask(item["question"])
    results.append((item, result))
    print(f"Q: {item['question']}")
    print(f"Expected: {item['category']} | Predicted: {result['category']} | "
          f"Confidence: {result['confidence']:.2f} | Escalation: {result['escalation_required']}")
    print(f"Answer: {result['answer']}")
    print(f"Sources: {[source['document'] for source in result['sources']]}\n")

labeled = [(item, result) for item, result in results if item["category"] != "unknown"]
category_accuracy = sum(item["category"] == result["category"] for item, result in labeled) / len(labeled)
print(f"Category accuracy: {category_accuracy:.1%} ({len(labeled)} labeled questions)")
print(f"Escalations: {sum(result['escalation_required'] for _, result in results)} / {len(results)}")